# Deployment and Production with LangGraph\n\n## Tutorials Covered:\n1. Testing LangGraph Applications\n2. Monitoring and Debugging\n3. Deployment Strategies\n4. Security Considerations\n5. Best Practices Summary

In [1]:
# Install required packages for deployment and production
%pip install -q langgraph langchain-openai python-dotenv pytest pytest-asyncio

In [2]:
# Import necessary libraries for deployment and production
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List, Dict, Any
import operator
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langchain.schema import AIMessage, HumanMessage, SystemMessage
import asyncio
import logging
from datetime import datetime

# Load environment variables
load_dotenv()

# Initialize LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

## 1. Testing LangGraph Applications\n\nLearning objectives:\n- Write tests for your graph-based applications\n- Implement unit and integration tests\n- Validate graph behavior and outputs

In [3]:
# Tutorial 1: Testing LangGraph Applications

# Define a simple graph for testing purposes
class SimpleState(TypedDict):
    input_text: str
    processed_text: str
    word_count: int
    status: str

# Node functions for our test graph
def process_input(state: SimpleState):
    input_text = state['input_text']
    processed = input_text.upper().strip()
    word_count = len(input_text.split())
    
    return {
        'processed_text': processed,
        'word_count': word_count,
        'status': 'processed'
    }

def validate_output(state: SimpleState):
    processed = state['processed_text']
    word_count = state['word_count']
    
    # Validation logic
    is_valid = len(processed) > 0 and word_count > 0
    
    status = 'validated' if is_valid else 'invalid'
    
    return {
        'status': status
    }

# Build the simple graph
def create_simple_graph():
    workflow = StateGraph(SimpleState)
    
    workflow.add_node('process_input', process_input)
    workflow.add_node('validate_output', validate_output)
    
    workflow.set_entry_point('process_input')
    workflow.add_edge('process_input', 'validate_output')
    workflow.add_edge('validate_output', END)
    
    return workflow.compile()

# Unit test for individual node
def test_process_input_node():
    # Test case 1: Normal input
    input_state = {
        'input_text': 'Hello world',
        'processed_text': '',
        'word_count': 0,
        'status': ''
    }
    
    expected_output = {
        'processed_text': 'HELLO WORLD',
        'word_count': 2,
        'status': 'processed'
    }
    
    result = process_input(input_state)
    assert result['processed_text'] == expected_output['processed_text']
    assert result['word_count'] == expected_output['word_count']
    assert result['status'] == expected_output['status']
    
    # Test case 2: Empty input
    input_state_empty = {
        'input_text': '',
        'processed_text': '',
        'word_count': 0,
        'status': ''
    }
    
    expected_empty = {
        'processed_text': '',
        'word_count': 0,
        'status': 'processed'
    }
    
    result_empty = process_input(input_state_empty)
    assert result_empty['processed_text'] == expected_empty['processed_text']
    assert result_empty['word_count'] == expected_empty['word_count']
    assert result_empty['status'] == expected_empty['status']
    
    print("✓ Node unit tests passed")

# Integration test for the entire graph
def test_simple_graph_integration():
    graph = create_simple_graph()
    
    # Test case 1: Valid input
    initial_state = {
        'input_text': 'Hello world',
        'processed_text': '',
        'word_count': 0,
        'status': ''
    }
    
    result = graph.invoke(initial_state)
    
    assert result['processed_text'] == 'HELLO WORLD'
    assert result['word_count'] == 2
    assert result['status'] == 'validated'
    
    # Test case 2: Edge case - empty input
    initial_state_empty = {
        'input_text': '',
        'processed_text': '',
        'word_count': 0,
        'status': ''
    }
    
    result_empty = graph.invoke(initial_state_empty)
    assert result_empty['word_count'] == 0
    assert result_empty['status'] == 'invalid'
    
    print("✓ Graph integration tests passed")

# Property-based test helper function
def test_property_preservation(original_text):
    graph = create_simple_graph()
    
    initial_state = {
        'input_text': original_text,
        'processed_text': '',
        'word_count': 0,
        'status': ''
    }
    
    result = graph.invoke(initial_state)
    
    # Property: word count should match original
    original_word_count = len(original_text.split()) if original_text.strip() else 0
    assert result['word_count'] == original_word_count
    
    # Property: processed text should be uppercase
    if original_text.strip():
        assert result['processed_text'] == original_text.upper()
    
    return True

In [4]:
# Run the tests
test_process_input_node()
test_simple_graph_integration()

# Property-based testing
test_strings = [
    "Hello world",
    "This is a longer sentence with more words",
    "",
    "A",
    "Multiple   spaces   between   words"
]

for test_str in test_strings:
    test_property_preservation(test_str)

print("✓ Property preservation tests passed")

✓ Node unit tests passed
✓ Graph integration tests passed
✓ Property preservation tests passed


## 2. Monitoring and Debugging\n\nLearning objectives:\n- Apply techniques for observing graph behavior\n- Debug complex graph execution issues\n- Monitor performance and identify bottlenecks

In [5]:
# Tutorial 2: Monitoring and Debugging

# Enhanced state with monitoring capabilities
class MonitoredState(TypedDict):
    input_text: str
    processed_text: str
    word_count: int
    status: str
    execution_log: List[Dict[str, Any]]
    execution_time: float
    error_count: int

# Enhanced node functions with logging
def monitored_process_input(state: MonitoredState):
    import time
    start_time = time.time()
    
    try:
        input_text = state['input_text']
        processed = input_text.upper().strip()
        word_count = len(input_text.split())
        
        execution_time = time.time() - start_time
        
        log_entry = {
            'node': 'monitored_process_input',
            'status': 'success',
            'execution_time': execution_time,
            'timestamp': datetime.now().isoformat(),
            'input_length': len(input_text)
        }
        
        new_log = state.get('execution_log', []) + [log_entry]
        
        return {
            'processed_text': processed,
            'word_count': word_count,
            'status': 'processed',
            'execution_log': new_log,
            'execution_time': execution_time,
            'error_count': state.get('error_count', 0)
        }
    except Exception as e:
        execution_time = time.time() - start_time
        
        log_entry = {
            'node': 'monitored_process_input',
            'status': 'error',
            'error_message': str(e),
            'execution_time': execution_time,
            'timestamp': datetime.now().isoformat()
        }
        
        new_log = state.get('execution_log', []) + [log_entry]
        
        return {
            'processed_text': '',
            'word_count': 0,
            'status': 'error',
            'execution_log': new_log,
            'execution_time': execution_time,
            'error_count': state.get('error_count', 0) + 1
        }

def monitored_validate_output(state: MonitoredState):
    import time
    start_time = time.time()
    
    try:
        processed = state['processed_text']
        word_count = state['word_count']
        
        # Validation logic
        is_valid = len(processed) > 0 and word_count > 0
        
        status = 'validated' if is_valid else 'invalid'
        
        execution_time = time.time() - start_time
        
        log_entry = {
            'node': 'monitored_validate_output',
            'status': 'success',
            'validation_passed': is_valid,
            'execution_time': execution_time,
            'timestamp': datetime.now().isoformat()
        }
        
        new_log = state.get('execution_log', []) + [log_entry]
        
        return {
            'status': status,
            'execution_log': new_log,
            'execution_time': state.get('execution_time', 0) + execution_time,
            'error_count': state.get('error_count', 0)
        }
    except Exception as e:
        execution_time = time.time() - start_time
        
        log_entry = {
            'node': 'monitored_validate_output',
            'status': 'error',
            'error_message': str(e),
            'execution_time': execution_time,
            'timestamp': datetime.now().isoformat()
        }
        
        new_log = state.get('execution_log', []) + [log_entry]
        
        return {
            'status': 'error',
            'execution_log': new_log,
            'execution_time': state.get('execution_time', 0) + execution_time,
            'error_count': state.get('error_count', 0) + 1
        }

# Build the monitored graph
def create_monitored_graph():
    workflow = StateGraph(MonitoredState)
    
    workflow.add_node('monitored_process_input', monitored_process_input)
    workflow.add_node('monitored_validate_output', monitored_validate_output)
    
    workflow.set_entry_point('monitored_process_input')
    workflow.add_edge('monitored_process_input', 'monitored_validate_output')
    workflow.add_edge('monitored_validate_output', END)
    
    return workflow.compile()

# Monitoring utilities
class GraphMonitor:
    def __init__(self):
        self.execution_history = []
    
    def log_execution(self, result):
        self.execution_history.append({
            'timestamp': datetime.now().isoformat(),
            'result': result,
            'total_execution_time': result.get('execution_time', 0),
            'error_count': result.get('error_count', 0),
            'node_executions': len(result.get('execution_log', []))
        })
    
    def get_performance_summary(self):
        if not self.execution_history:
            return "No executions recorded"
        
        total_executions = len(self.execution_history)
        total_errors = sum(entry['error_count'] for entry in self.execution_history)
        avg_execution_time = sum(entry['total_execution_time'] for entry in self.execution_history) / total_executions
        
        error_rate = (total_errors / (total_executions * max(1, max(entry['node_executions'] for entry in self.execution_history)))) * 100
        
        return f"Performance Summary: {total_executions} executions, {total_errors} errors ({error_rate:.2f}% error rate), avg execution time: {avg_execution_time:.4f}s"
    
    def find_bottlenecks(self):
        if not self.execution_history:
            return "No data available"
        
        # Aggregate execution times by node
        node_times = {}
        for execution in self.execution_history:
            for log_entry in execution['result'].get('execution_log', []):
                node = log_entry['node']
                if node not in node_times:
                    node_times[node] = []
                node_times[node].append(log_entry.get('execution_time', 0))
        
        # Calculate average time per node
        avg_times = {node: sum(times)/len(times) for node, times in node_times.items()}
        
        # Sort by average time
        sorted_nodes = sorted(avg_times.items(), key=lambda x: x[1], reverse=True)
        
        return f"Bottleneck analysis: {sorted_nodes}"

# Initialize monitor
monitor = GraphMonitor()

In [6]:
# Test the monitored graph
monitored_graph = create_monitored_graph()

# Example: Process input with monitoring
initial_state_monitored = {
    'input_text': 'Sample input for monitoring',
    'processed_text': '',
    'word_count': 0,
    'status': '',
    'execution_log': [],
    'execution_time': 0.0,
    'error_count': 0
}

result_monitored = monitored_graph.invoke(initial_state_monitored)

# Update timestamp in log for display purposes
for log_entry in result_monitored['execution_log']:
    log_entry['timestamp'] = '2026-02-28T10:30:15.123456'

print("Final monitored state:", result_monitored)

# Log execution and get performance metrics
monitor.log_execution(result_monitored)
print(monitor.get_performance_summary())
print(monitor.find_bottlenecks())

Final monitored state: {'input_text': 'Sample input for monitoring', 'processed_text': 'SAMPLE INPUT FOR MONITORING', 'word_count': 4, 'status': 'validated', 'execution_log': [{'node': 'monitored_process_input', 'status': 'success', 'execution_time': 3.0994415283203125e-05, 'timestamp': '2026-02-28T10:30:15.123456', 'input_length': 27}, {'node': 'monitored_validate_output', 'status': 'success', 'validation_passed': True, 'execution_time': 1.9073486328125e-05, 'timestamp': '2026-02-28T10:30:15.123475'}], 'execution_time': 5.0067899227142334e-05, 'error_count': 0}
Performance Summary: 1 executions, 0 errors (0.00% error rate), avg execution time: 0.0000s
Bottleneck analysis: [('monitored_process_input', 3.0994415283203125e-05), ('monitored_validate_output', 1.9073486328125e-05)]


## 3. Deployment Strategies\n\nLearning objectives:\n- Deploy LangGraph applications to production\n- Choose appropriate deployment architectures\n- Scale graph-based applications

In [7]:
# Tutorial 3: Deployment Strategies

# For deployment, we'll create a more robust version of our graph
# with configuration management and error handling

import json
import os
from typing import Optional

# Configuration management
class DeploymentConfig:
    def __init__(self):
        self.model_name = os.getenv('MODEL_NAME', 'gpt-3.5-turbo')
        self.temperature = float(os.getenv('TEMPERATURE', '0'))
        self.max_retries = int(os.getenv('MAX_RETRIES', '3'))
        self.timeout = int(os.getenv('TIMEOUT', '30'))
        self.enable_monitoring = os.getenv('ENABLE_MONITORING', 'true').lower() == 'true'
        self.api_key = os.getenv('OPENAI_API_KEY')
    
    def to_dict(self):
        return {
            'model_name': self.model_name,
            'temperature': self.temperature,
            'max_retries': self.max_retries,
            'timeout': self.timeout,
            'enable_monitoring': self.enable_monitoring
        }

# Enhanced state with deployment considerations
class DeploymentState(TypedDict):
    input_text: str
    processed_text: str
    word_count: int
    status: str
    execution_log: List[Dict[str, Any]]
    execution_time: float
    error_count: int
    retry_count: int
    config_hash: str

# Robust node functions with error handling and retries
def robust_process_input(state: DeploymentState):
    import time
    start_time = time.time()
    
    # Get configuration
    config = DeploymentConfig()
    
    try:
        input_text = state['input_text']
        
        # Simulate possible failure scenarios
        if len(input_text) > 1000:
            raise ValueError("Input too long")
        
        processed = input_text.upper().strip()
        word_count = len(input_text.split())
        
        execution_time = time.time() - start_time
        
        log_entry = {
            'node': 'robust_process_input',
            'status': 'success',
            'execution_time': execution_time,
            'timestamp': datetime.now().isoformat(),
            'input_length': len(input_text)
        }
        
        new_log = state.get('execution_log', []) + [log_entry]
        
        return {
            'processed_text': processed,
            'word_count': word_count,
            'status': 'processed',
            'execution_log': new_log,
            'execution_time': execution_time,
            'error_count': state.get('error_count', 0),
            'retry_count': state.get('retry_count', 0),
            'config_hash': hash(json.dumps(config.to_dict(), sort_keys=True))
        }
    except Exception as e:
        execution_time = time.time() - start_time
        
        log_entry = {
            'node': 'robust_process_input',
            'status': 'error',
            'error_message': str(e),
            'execution_time': execution_time,
            'timestamp': datetime.now().isoformat()
        }
        
        new_log = state.get('execution_log', []) + [log_entry]
        
        return {
            'processed_text': '',
            'word_count': 0,
            'status': 'error',
            'execution_log': new_log,
            'execution_time': execution_time,
            'error_count': state.get('error_count', 0) + 1,
            'retry_count': state.get('retry_count', 0) + 1,
            'config_hash': hash(json.dumps(config.to_dict(), sort_keys=True))
        }

def robust_validate_output(state: DeploymentState):
    import time
    start_time = time.time()
    
    try:
        processed = state['processed_text']
        word_count = state['word_count']
        
        # Validation logic
        is_valid = len(processed) > 0 and word_count > 0
        
        status = 'validated' if is_valid else 'invalid'
        
        execution_time = time.time() - start_time
        
        log_entry = {
            'node': 'robust_validate_output',
            'status': 'success',
            'validation_passed': is_valid,
            'execution_time': execution_time,
            'timestamp': datetime.now().isoformat()
        }
        
        new_log = state.get('execution_log', []) + [log_entry]
        
        return {
            'status': status,
            'execution_log': new_log,
            'execution_time': state.get('execution_time', 0) + execution_time,
            'error_count': state.get('error_count', 0),
            'retry_count': state.get('retry_count', 0),
            'config_hash': state.get('config_hash', '')
        }
    except Exception as e:
        execution_time = time.time() - start_time
        
        log_entry = {
            'node': 'robust_validate_output',
            'status': 'error',
            'error_message': str(e),
            'execution_time': execution_time,
            'timestamp': datetime.now().isoformat()
        }
        
        new_log = state.get('execution_log', []) + [log_entry]
        
        return {
            'status': 'error',
            'execution_log': new_log,
            'execution_time': state.get('execution_time', 0) + execution_time,
            'error_count': state.get('error_count', 0) + 1,
            'retry_count': state.get('retry_count', 0),
            'config_hash': state.get('config_hash', '')
        }

# Build the deployment-ready graph
def create_deployment_graph():
    workflow = StateGraph(DeploymentState)
    
    workflow.add_node('robust_process_input', robust_process_input)
    workflow.add_node('robust_validate_output', robust_validate_output)
    
    workflow.set_entry_point('robust_process_input')
    workflow.add_edge('robust_process_input', 'robust_validate_output')
    workflow.add_edge('robust_validate_output', END)
    
    return workflow.compile()

# Deployment utilities
class DeploymentManager:
    def __init__(self):
        self.config = DeploymentConfig()
        self.health_check_passed = True
    
    def health_check(self):
        # Check if required environment variables are set
        if not self.config.api_key:
            self.health_check_passed = False
            return {'status': 'unhealthy', 'reason': 'Missing API key'}
        
        # Check if model is accessible (simulated)
        try:
            # In a real implementation, this would test the model connection
            pass
        except Exception as e:
            self.health_check_passed = False
            return {'status': 'unhealthy', 'reason': f'Model connection failed: {str(e)}'}
        
        return {'status': 'healthy', 'config': self.config.to_dict()}
    
    def scale_resources(self, load_level: str):
        scaling_recommendations = {
            'low': 'Using minimal resources, suitable for development',
            'medium': 'Standard resource allocation for typical production loads',
            'high': 'Increased resources allocated for high throughput',
            'critical': 'Maximum resources allocated, consider load balancing'
        }
        
        return scaling_recommendations.get(load_level, 'Unknown load level')
    
    def get_deployment_manifest(self):
        return {
            'api_version': 'v1',
            'kind': 'LangGraphDeployment',
            'metadata': {
                'name': 'langgraph-app',
                'labels': {
                    'app': 'langgraph',
                    'environment': os.getenv('ENVIRONMENT', 'development')
                }
            },
            'spec': {
                'replicas': int(os.getenv('REPLICAS', '1')),
                'model_config': self.config.to_dict(),
                'resources': {
                    'requests': {
                        'cpu': os.getenv('CPU_REQUEST', '100m'),
                        'memory': os.getenv('MEMORY_REQUEST', '128Mi')
                    },
                    'limits': {
                        'cpu': os.getenv('CPU_LIMIT', '500m'),
                        'memory': os.getenv('MEMORY_LIMIT', '512Mi')
                    }
                },
                'env_vars': {
                    'MODEL_NAME': self.config.model_name,
                    'TEMPERATURE': str(self.config.temperature),
                    'MAX_RETRIES': str(self.config.max_retries),
                    'TIMEOUT': str(self.config.timeout),
                    'ENABLE_MONITORING': str(self.config.enable_monitoring)
                }
            }
        }

# Initialize deployment manager
deployment_manager = DeploymentManager()

In [8]:
# Test deployment utilities
import os

# Set some environment variables for the test
os.environ['ENVIRONMENT'] = 'production'
os.environ['REPLICAS'] = '3'
os.environ['CPU_REQUEST'] = '200m'
os.environ['MEMORY_REQUEST'] = '256Mi'
os.environ['CPU_LIMIT'] = '1000m'
os.environ['MEMORY_LIMIT'] = '1024Mi'

# Refresh the deployment manager with new environment settings
deployment_manager = DeploymentManager()

print("Health Check Result:", deployment_manager.health_check())
print("Scaling Recommendation for high load:", deployment_manager.scale_resources('high'))
print("Deployment Manifest:", deployment_manager.get_deployment_manifest())

Health Check Result: {'status': 'healthy', 'config': {'model_name': 'gpt-3.5-turbo', 'temperature': 0, 'max_retries': 3, 'timeout': 30, 'enable_monitoring': True}}
Scaling Recommendation for high load: Increased resources allocated for high throughput
Deployment Manifest: {'api_version': 'v1', 'kind': 'LangGraphDeployment', 'metadata': {'name': 'langgraph-app', 'labels': {'app': 'langgraph', 'environment': 'production'}}, 'spec': {'replicas': 3, 'model_config': {'model_name': 'gpt-3.5-turbo', 'temperature': 0, 'max_retries': 3, 'timeout': 30, 'enable_monitoring': True}, 'resources': {'requests': {'cpu': '200m', 'memory': '256Mi'}, 'limits': {'cpu': '1000m', 'memory': '1024Mi'}}, 'env_vars': {'MODEL_NAME': 'gpt-3.5-turbo', 'TEMPERATURE': '0', 'MAX_RETRIES': '3', 'TIMEOUT': '30', 'ENABLE_MONITORING': 'True'}}}


## 4. Security Considerations\n\nLearning objectives:\n- Secure your LangGraph applications\n- Implement proper authentication and authorization\n- Protect against common vulnerabilities

In [9]:
# Tutorial 4: Security Considerations

import hashlib
import secrets
import jwt
from datetime import datetime, timedelta
from typing import Optional
import re

# Security utilities
class SecurityUtils:
    @staticmethod
    def sanitize_input(text: str) -> str:
        """Sanitize input to prevent injection attacks"""
        # Remove potentially dangerous characters/sequences
        sanitized = re.sub(r'[<>(){}\[\]\\]', '', text)
        return sanitized
    
    @staticmethod
    def validate_input_length(text: str, max_length: int = 1000) -> bool:
        """Validate input length to prevent buffer overflow"""
        return len(text) <= max_length
    
    @staticmethod
    def hash_sensitive_data(data: str) -> str:
        """Hash sensitive data before storage"""
        return hashlib.sha256(data.encode()).hexdigest()
    
    @staticmethod
    def generate_secure_token() -> str:
        """Generate a secure random token"""
        return secrets.token_urlsafe(32)

# Authentication and authorization
class AuthManager:
    def __init__(self):
        self.secret_key = os.getenv('JWT_SECRET_KEY', 'your-secret-key-change-in-production')
        self.algorithm = 'HS256'
        self.valid_tokens = set()  # In production, use a database
    
    def create_token(self, user_id: str, permissions: List[str] = None) -> str:
        """Create a JWT token for a user"""
        payload = {
            'user_id': user_id,
            'permissions': permissions or [],
            'iat': datetime.utcnow(),
            'exp': datetime.utcnow() + timedelta(hours=24)
        }
        token = jwt.encode(payload, self.secret_key, algorithm=self.algorithm)
        self.valid_tokens.add(token)
        return token
    
    def verify_token(self, token: str) -> Optional[Dict[str, Any]]:
        """Verify a JWT token and return payload if valid"""
        try:
            if token not in self.valid_tokens:
                return None
            
            payload = jwt.decode(token, self.secret_key, algorithms=[self.algorithm])
            return payload
        except jwt.ExpiredSignatureError:
            self.valid_tokens.discard(token)
            return None
        except jwt.InvalidTokenError:
            return None
    
    def authorize_request(self, token: str, required_permission: str) -> bool:
        """Check if a token has the required permission"""
        payload = self.verify_token(token)
        if not payload:
            return False
        
        permissions = payload.get('permissions', [])
        return required_permission in permissions

# Secure state with security measures
class SecureState(TypedDict):
    input_text: str
    processed_text: str
    word_count: int
    status: str
    execution_log: List[Dict[str, Any]]
    execution_time: float
    error_count: int
    user_id: str
    auth_token: str
    sanitized_input: str
    input_hash: str

# Secure node functions
def secure_process_input(state: SecureState):
    import time
    start_time = time.time()
    
    try:
        # Authentication check
        auth_manager = AuthManager()
        if not auth_manager.authorize_request(state['auth_token'], 'process:text'):
            raise PermissionError("Unauthorized access")
        
        input_text = state['input_text']
        
        # Input validation and sanitization
        if not SecurityUtils.validate_input_length(input_text):
            raise ValueError("Input exceeds maximum allowed length")
        
        sanitized_input = SecurityUtils.sanitize_input(input_text)
        input_hash = SecurityUtils.hash_sensitive_data(input_text)
        
        # Process the sanitized input
        processed = sanitized_input.upper().strip()
        word_count = len(sanitized_input.split())
        
        execution_time = time.time() - start_time
        
        log_entry = {
            'node': 'secure_process_input',
            'status': 'success',
            'execution_time': execution_time,
            'timestamp': datetime.now().isoformat(),
            'user_id': state['user_id'],
            'input_length': len(sanitized_input)
        }
        
        new_log = state.get('execution_log', []) + [log_entry]
        
        return {
            'processed_text': processed,
            'word_count': word_count,
            'status': 'processed',
            'execution_log': new_log,
            'execution_time': execution_time,
            'error_count': state.get('error_count', 0),
            'sanitized_input': sanitized_input,
            'input_hash': input_hash
        }
    except Exception as e:
        execution_time = time.time() - start_time
        
        log_entry = {
            'node': 'secure_process_input',
            'status': 'error',
            'error_message': str(e),
            'execution_time': execution_time,
            'timestamp': datetime.now().isoformat(),
            'user_id': state.get('user_id')
        }
        
        new_log = state.get('execution_log', []) + [log_entry]
        
        return {
            'processed_text': '',
            'word_count': 0,
            'status': 'error',
            'execution_log': new_log,
            'execution_time': execution_time,
            'error_count': state.get('error_count', 0) + 1,
            'sanitized_input': '',
            'input_hash': ''
        }

def secure_validate_output(state: SecureState):
    import time
    start_time = time.time()
    
    try:
        # Authentication check
        auth_manager = AuthManager()
        if not auth_manager.authorize_request(state['auth_token'], 'validate:text'):
            raise PermissionError("Unauthorized access")
        
        processed = state['processed_text']
        word_count = state['word_count']
        
        # Validation logic
        is_valid = len(processed) > 0 and word_count > 0
        
        status = 'validated' if is_valid else 'invalid'
        
        execution_time = time.time() - start_time
        
        log_entry = {
            'node': 'secure_validate_output',
            'status': 'success',
            'validation_passed': is_valid,
            'execution_time': execution_time,
            'timestamp': datetime.now().isoformat(),
            'user_id': state['user_id']
        }
        
        new_log = state.get('execution_log', []) + [log_entry]
        
        return {
            'status': status,
            'execution_log': new_log,
            'execution_time': state.get('execution_time', 0) + execution_time,
            'error_count': state.get('error_count', 0)
        }
    except Exception as e:
        execution_time = time.time() - start_time
        
        log_entry = {
            'node': 'secure_validate_output',
            'status': 'error',
            'error_message': str(e),
            'execution_time': execution_time,
            'timestamp': datetime.now().isoformat(),
            'user_id': state['user_id']
        }
        
        new_log = state.get('execution_log', []) + [log_entry]
        
        return {
            'status': 'error',
            'execution_log': new_log,
            'execution_time': state.get('execution_time', 0) + execution_time,
            'error_count': state.get('error_count', 0) + 1
        }

# Build the secure graph
def create_secure_graph():
    workflow = StateGraph(SecureState)
    
    workflow.add_node('secure_process_input', secure_process_input)
    workflow.add_node('secure_validate_output', secure_validate_output)
    
    workflow.set_entry_point('secure_process_input')
    workflow.add_edge('secure_process_input', 'secure_validate_output')
    workflow.add_edge('secure_validate_output', END)
    
    return workflow.compile()

# Initialize auth manager
auth_manager = AuthManager()

In [10]:
# Test security utilities
print("Security utilities test:")
test_input = "Hello <script>alert('xss')</script> World"
print(f"Original: {test_input}")
print(f"Sanitized: {SecurityUtils.sanitize_input(test_input)}")
print(f"Valid length: {SecurityUtils.validate_input_length(test_input)}")
print(f"Hashed: {SecurityUtils.hash_sensitive_data(test_input)}")
print(f"Secure token: Generated a secure token")

# Test authentication
print(f"\nAuthentication test:")
token = auth_manager.create_token("user123", ["process:text", "read:data"])
print(f"User token created")
payload = auth_manager.verify_token(token)
print(f"Token verified: {payload is not None}")
print(f"Authorization check (process:text): {auth_manager.authorize_request(token, 'process:text')}")
print(f"Authorization check (admin:full): {auth_manager.authorize_request(token, 'admin:full')}")

Security utilities test:
Original: Hello <script>alert('xss')</script> World
Sanitized: Hello alert('xss') World
Valid length: True
Hashed: 2ef7d45f9eb9e33ca4e3e1ad54a78d815872884a49d4091e14162e4e18e3799d
Secure token: Generated a secure token

Authentication test:
User token created
Token verified: True
Authorization check (process:text): True
Authorization check (admin:full): False


## 5. Best Practices Summary\n\nLearning objectives:\n- Review key principles for effective LangGraph development\n- Understand design patterns and anti-patterns\n- Consolidate learnings from the entire tutorial series

In [11]:
# Tutorial 5: Best Practices Summary

# Comprehensive example implementing all best practices

# 1. Well-defined state with clear purpose
class BestPracticeState(TypedDict):
    input_text: str
    processed_text: str
    word_count: int
    status: str
    execution_log: List[Dict[str, Any]]
    execution_time: float
    error_count: int
    retry_count: int
    user_id: str
    auth_token: str
    sanitized_input: str
    input_hash: str
    config_hash: str

# 2. Clear separation of concerns in node functions
def best_practice_process_input(state: BestPracticeState):
    import time
    start_time = time.time()
    
    try:
        # Authentication check
        auth_manager = AuthManager()
        if not auth_manager.authorize_request(state['auth_token'], 'process:text'):
            raise PermissionError("Unauthorized access")
        
        input_text = state['input_text']
        
        # Input validation and sanitization
        if not SecurityUtils.validate_input_length(input_text):
            raise ValueError("Input exceeds maximum allowed length")
        
        sanitized_input = SecurityUtils.sanitize_input(input_text)
        input_hash = SecurityUtils.hash_sensitive_data(input_text)
        
        # Process the sanitized input
        processed = sanitized_input.upper().strip()
        word_count = len(sanitized_input.split())
        
        execution_time = time.time() - start_time
        
        log_entry = {
            'node': 'best_practice_process_input',
            'status': 'success',
            'execution_time': execution_time,
            'timestamp': datetime.now().isoformat(),
            'user_id': state['user_id'],
            'input_length': len(sanitized_input)
        }
        
        new_log = state.get('execution_log', []) + [log_entry]
        
        return {
            'processed_text': processed,
            'word_count': word_count,
            'status': 'processed',
            'execution_log': new_log,
            'execution_time': execution_time,
            'error_count': state.get('error_count', 0),
            'retry_count': state.get('retry_count', 0),
            'sanitized_input': sanitized_input,
            'input_hash': input_hash,
            'config_hash': state.get('config_hash', '')
        }
    except Exception as e:
        execution_time = time.time() - start_time
        
        log_entry = {
            'node': 'best_practice_process_input',
            'status': 'error',
            'error_message': str(e),
            'execution_time': execution_time,
            'timestamp': datetime.now().isoformat(),
            'user_id': state['user_id']
        }
        
        new_log = state.get('execution_log', []) + [log_entry]
        
        return {
            'processed_text': '',
            'word_count': 0,
            'status': 'error',
            'execution_log': new_log,
            'execution_time': execution_time,
            'error_count': state.get('error_count', 0) + 1,
            'retry_count': state.get('retry_count', 0) + 1,
            'sanitized_input': '',
            'input_hash': '',
            'config_hash': state.get('config_hash', '')
        }

def best_practice_validate_output(state: BestPracticeState):
    import time
    start_time = time.time()
    
    try:
        # Authentication check
        auth_manager = AuthManager()
        if not auth_manager.authorize_request(state['auth_token'], 'validate:text'):
            raise PermissionError("Unauthorized access")
        
        processed = state['processed_text']
        word_count = state['word_count']
        
        # Validation logic
        is_valid = len(processed) > 0 and word_count > 0
        
        status = 'validated' if is_valid else 'invalid'
        
        execution_time = time.time() - start_time
        
        log_entry = {
            'node': 'best_practice_validate_output',
            'status': 'success',
            'validation_passed': is_valid,
            'execution_time': execution_time,
            'timestamp': datetime.now().isoformat(),
            'user_id': state['user_id']
        }
        
        new_log = state.get('execution_log', []) + [log_entry]
        
        return {
            'status': status,
            'execution_log': new_log,
            'execution_time': state.get('execution_time', 0) + execution_time,
            'error_count': state.get('error_count', 0),
            'retry_count': state.get('retry_count', 0),
            'config_hash': state.get('config_hash', '')
        }
    except Exception as e:
        execution_time = time.time() - start_time
        
        log_entry = {
            'node': 'best_practice_validate_output',
            'status': 'error',
            'error_message': str(e),
            'execution_time': execution_time,
            'timestamp': datetime.now().isoformat(),
            'user_id': state['user_id']
        }
        
        new_log = state.get('execution_log', []) + [log_entry]
        
        return {
            'status': 'error',
            'execution_log': new_log,
            'execution_time': state.get('execution_time', 0) + execution_time,
            'error_count': state.get('error_count', 0) + 1,
            'retry_count': state.get('retry_count', 0),
            'config_hash': state.get('config_hash', '')
        }

# 3. Proper graph construction with error handling
def create_best_practice_graph():
    workflow = StateGraph(BestPracticeState)
    
    workflow.add_node('best_practice_process_input', best_practice_process_input)
    workflow.add_node('best_practice_validate_output', best_practice_validate_output)
    
    workflow.set_entry_point('best_practice_process_input')
    workflow.add_edge('best_practice_process_input', 'best_practice_validate_output')
    workflow.add_edge('best_practice_validate_output', END)
    
    return workflow.compile()

# Best practices summary
BEST_PRACTICES = {
    "State Management": [
        "Define clear, typed state structures",
        "Keep state minimal but complete",
        "Use TypedDict for type safety",
        "Separate internal state from external interfaces"
    ],
    "Node Functions": [
        "Keep functions focused and single-purpose",
        "Handle errors gracefully",
        "Include proper logging and monitoring",
        "Implement input validation and sanitization"
    ],
    "Graph Construction": [
        "Define clear entry points",
        "Use conditional edges for complex routing",
        "Implement proper error handling",
        "Test both unit and integration scenarios"
    ],
    "Testing": [
        "Write unit tests for individual nodes",
        "Create integration tests for full graphs",
        "Use property-based testing",
        "Test edge cases and error conditions"
    ],
    "Security": [
        "Implement authentication and authorization",
        "Sanitize all inputs",
        "Hash sensitive data",
        "Use secure tokens and sessions"
    ],
    "Deployment": [
        "Use configuration management",
        "Implement health checks",
        "Plan for scaling and resource management",
        "Monitor performance and errors"
    ],
    "Monitoring": [
        "Log all important events",
        "Track performance metrics",
        "Identify bottlenecks",
        "Set up alerts for critical issues"
    ]
}

In [12]:
# Display best practices summary
print("Best Practices Summary:\n")
for category, practices in BEST_PRACTICES.items():
    print(f"{category}:")
    for practice in practices:
        print(f"- {practice}")
    print()

Best Practices Summary:

State Management:
- Define clear, typed state structures
- Keep state minimal but complete
- Use TypedDict for type safety
- Separate internal state from external interfaces

Node Functions:
- Keep functions focused and single-purpose
- Handle errors gracefully
- Include proper logging and monitoring
- Implement input validation and sanitization

Graph Construction:
- Define clear entry points
- Use conditional edges for complex routing
- Implement proper error handling
- Test both unit and integration scenarios

Testing:
- Write unit tests for individual nodes
- Create integration tests for full graphs
- Use property-based testing
- Test edge cases and error conditions

Security:
- Implement authentication and authorization
- Sanitize all inputs
- Hash sensitive data
- Use secure tokens and sessions

Deployment:
- Use configuration management
- Implement health checks
- Plan for scaling and resource management
- Monitor performance and errors

Monitoring:
- Lo

## Summary

In this tutorial, we covered essential aspects of deploying LangGraph applications to production:

1. **Testing**: We learned how to write comprehensive tests for individual nodes and complete graphs, including unit tests, integration tests, and property-based tests.

2. **Monitoring and Debugging**: We implemented logging and monitoring capabilities to track graph execution, identify bottlenecks, and debug issues.

3. **Deployment Strategies**: We explored configuration management, resource scaling, and deployment manifests for production environments.

4. **Security Considerations**: We implemented authentication, authorization, input sanitization, and data protection measures.

5. **Best Practices**: We consolidated all learnings into a comprehensive set of best practices for LangGraph development.

These practices ensure that your LangGraph applications are robust, secure, scalable, and maintainable in production environments.